In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from mlp_manual.mlp import MLP,Linear,ReLU, SoftmaxCrossEntropy
from mlp_manual.Sampler import Sampler
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
def show_images(dataset, num_images=4):
    plt.figure(figsize=(10, 5))
    for i in range(num_images):
        img_tensor, label = dataset[i] 
        inv_img = (img_tensor * 0.229) + 0.485 
        img_np = inv_img.squeeze().numpy() 
        plt.subplot(1, num_images, i+1)
        plt.imshow(img_np, cmap='gray')
        plt.title(f"Label: {label}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()


def count_classes(dataset, name="Dataset"):
    if hasattr(dataset, 'targets'):
        all_labels = dataset.targets
    else:
        all_labels = []
        for data, label in dataset:
            all_labels.append(label)
    counter = Counter(all_labels)
    print(f"--- {name} 统计结果 ---")
    for class_idx in sorted(counter.keys()):
        print(f"类别 {class_idx}: {counter[class_idx]} 张")

### 1. 宠物数据集(37类)

In [ ]:
root = "../data"
transform = transforms.Compose([
    # transforms.Resize((8, 8)),
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Normalize(
        mean=[0.485], std=[0.229]   # 针对单通道灰度图的 ImageNet 均值和标准差
    )
])

train_dataset = datasets.OxfordIIITPet(
    root=root, 
    split='trainval', 
    download=True, 
    transform=transform
)

test_dataset = datasets.OxfordIIITPet(
    root=root, 
    split='test', 
    download=True, 
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

In [ ]:
show_images(test_dataset, num_images=4)

In [ ]:
show_images(test_dataset, num_images=4)

In [ ]:
# 4. 执行统计
count_classes(train_dataset, "训练集 (Train+Val)")
count_classes(test_dataset, "测试集 (Test)")

In [ ]:
for imgs, labels in train_loader:
    print(imgs.shape,labels.shape)  # 应该是 [32, 3, H, W]
    break

### 2.花卉数据集

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""


# 数据准备：从 data/MNIST 中读取，缩放到 8x8，取 5000 训练 + 1000 测试
root = "../data"  # 数据文件夹，包含 MNIST 子目录
transform = transforms.Compose([
    transforms.Resize((8, 8)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_full = datasets.MNIST(root=root, train=True, download=True, transform=transform)
test_full = datasets.MNIST(root=root, train=False, download=True, transform=transform)

train_ds = Subset(train_full, list(range(5000)))
test_ds = Subset(test_full, list(range(1000)))

batch_size = 16
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)





In [ ]:
layer1_info = {
    "forward":"software",                           # software or hardware
    "chip":None,
    "row_index":[i for i in range(64) ],
    "col_index":[i for i in range(64) ],
    "from_row":True,
    "quantization":"noShift",                       # 其中noShift量化，每次推理结果的系数因子相同；shift量化，每次推理结果的系数因子不同，需要移位相加
    "quantization_bits":8,
    "cond_to_weight_scale":0.1,                     # 电导到权重的缩放比例
    "cond_to_weight_method":"reference",            # reference or difference
    "reference_cond":550,                           # 参考电导值


    "update":"software",                            # software or hardware
    "lr":0.01,
    "BL":0,
    "cond_to_pulse_width":1,                    # 电导到脉宽变化的比例
    "write_voltage":3,                              # 写入电压
    
}


layer2_info = {
    "forward":"software",                           # software or hardware
    "chip":None,
    "row_index":[i for i in range(64,128) ],
    "col_index":[i for i in range(10) ],
    "from_row":True,
    "quantization":"noShift",                       # 其中noShift量化，每次推理结果的系数因子相同；shift量化，每次推理结果的系数因子不同，需要移位相加
    "quantization_bits":8,
    "cond_to_weight_scale":0.1,                     # 电导到权重的缩放比例
    "cond_to_weight_method":"reference",            # reference or difference
    "reference_cond":550,                           # 参考电导值


    "update":"software",                            # software or hardware
    "lr":0.01,
    "BL":0,
    "cond_to_pulse_width":1,                    # 电导到脉宽变化的比例
    "write_voltage":3,                              # 写入电压

    "sampler": Sampler(mode="sobol",sliding=True)
    
}

# 训练超参
epochs = 30

In [ ]:
model = MLP()
model.append(Linear(64, 64,Sampler(mode="sobol",sliding=True),layer1_info))
model.append(ReLU())
model.append(Linear(64, 37,Sampler(mode="sobol",sliding=True),layer2_info))
criterion = SoftmaxCrossEntropy()


for epoch in range(1, epochs + 1):
    total_loss = 0.0
    n_samples = 0
    for imgs, labels in train_loader:
        x = imgs.numpy().reshape(imgs.shape[0], -1)
        y = labels.numpy().astype(np.int64)

        logits = model.forward(x)
        loss = criterion.forward(logits, y)
        total_loss += float(loss) * x.shape[0]
        n_samples += x.shape[0]

        grad_logits = criterion.backward()
        _ = model.backward(grad_logits)

    avg_loss = total_loss / n_samples



    correct = 0
    total = 0
    for imgs, labels in test_loader:
        x = imgs.numpy().reshape(imgs.shape[0], -1)
        logits = model.forward(x)
        preds = np.argmax(logits, axis=1)
        correct += int((preds == labels.numpy()).sum())
        total += imgs.shape[0]

    acc = correct / total
    print(f"Epoch {epoch}/{epochs}  loss={avg_loss:.4f}  test_acc={acc:.4f}")

In [ ]:
model = MLP()
model.append(Linear(64, 64,Sampler(mode="sobol",sliding=True),layer1_info))
model.append(ReLU())
model.append(Linear(64, 37,Sampler(mode="sobol",sliding=True),layer2_info))
criterion = SoftmaxCrossEntropy()
from sklearn.metrics import classification_report

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    n_samples = 0

    for imgs, labels in train_loader:
        x = imgs.numpy().reshape(imgs.shape[0], -1)
        y = labels.numpy().astype(np.int64)

        logits = model.forward(x)
        loss = criterion.forward(logits, y)
        total_loss += float(loss) * x.shape[0]
        n_samples += x.shape[0]

        grad_logits = criterion.backward()
        _ = model.backward(grad_logits)

    avg_loss = total_loss / n_samples


    

    # ... 之前的训练代码 ...

    # 初始化用于存储所有预测结果和真实标签的列表
    all_preds = []
    all_labels = []


    for imgs, labels in test_loader:
        x = imgs.numpy().reshape(imgs.shape[0], -1)
        logits = model.forward(x)
        preds = np.argmax(logits, axis=1)

        # 收集结果
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    target_names = train_dataset.classes # 假设你的 Dataset 对象有 classes 属性

    print("\n" + "="*60)
    print("          各类别分类性能报告 (Classification Report)")
    print("="*60)
    report = classification_report(all_labels, all_preds, target_names=target_names)
    print(report)

In [ ]:
import numpy as np

# 假设你的类别数量是 37 (Oxford-IIIT Pet 分类任务)
num_classes = 37
class_correct = np.zeros(num_classes)
class_total = np.zeros(num_classes)

# 如果你的数据集对象有 class_to_idx 属性，可以用来反向映射名称
# class_names = {v: k for k, v in train_dataset.class_to_idx.items()}

for imgs, labels in test_loader:
    x = imgs.numpy().reshape(imgs.shape[0], -1)
    logits = model.forward(x)
    preds = np.argmax(logits, axis=1)

    # 转换为 numpy 数组以便索引
    labels_np = labels.numpy()
    preds_np = preds

    # 遍历每个类别
    for i in range(num_classes):
        # 创建掩码：找出真实标签中所有属于第 i 类的位置
        mask = (labels_np == i)
        # 统计该批次中第 i 类的总样本数
        class_total[i] += np.sum(mask)
        # 统计该批次中第 i 类被正确预测的数量
        # mask 也用于筛选 preds_np，看对应位置是否等于 i
        class_correct[i] += np.sum(preds_np[mask] == i)

print("\n" + "="*40)
print("          各类别准确率 (Class-wise Accuracy)")
print("="*40)
for i in range(num_classes):
    if class_total[i] > 0:
        acc = 100 * class_correct[i] / class_total[i]
        # print(f"Class {i:2d} : {acc:.2f}% ({int(class_total[i])} samples)")
        # 如果有类别名称映射，可以打印名字：
        print(f"Class {i} : {acc:.2f}%")
    else:
        print(f"Class {i} : No samples found in test set.")

In [ ]:

model = MLP([in_dim, out_dim], activation="relu",sampler_mode="sobol",sliding=True)  # 单层 MLP
criterion = SoftmaxCrossEntropy()


for epoch in range(1, epochs + 1):
    total_loss = 0.0
    n_samples = 0
    for imgs, labels in train_loader:
        x = imgs.numpy().astype(DTYPE).reshape(imgs.shape[0], -1)
        y = labels.numpy().astype(np.int64)

        logits = model.forward(x)
        loss = criterion.forward(logits, y)
        total_loss += float(loss) * x.shape[0]
        n_samples += x.shape[0]

        grad_logits = criterion.backward()
        _ = model.backward(grad_logits, update_info={'lr': lr})

    avg_loss = total_loss / n_samples

    correct = 0
    total = 0
    for imgs, labels in test_loader:
        x = imgs.numpy().reshape(imgs.shape[0], -1)
        logits = model.forward(x)
        preds = np.argmax(logits, axis=1)
        correct += int((preds == labels.numpy()).sum())
        total += imgs.shape[0]

    acc = correct / total
    print(f"Epoch {epoch}/{epochs}  loss={avg_loss:.4f}  test_acc={acc:.4f}")